# 01 — Exploratory Data Analysis
NASA CMAPSS Turbofan Engine Degradation Dataset — FD001

Goals:
- Load and inspect raw data
- Identify constant / near-zero variance sensors to drop
- Understand RUL distribution
- Plot sensor degradation trends over engine lifetime

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
DATA_DIR = Path('../data/raw')

In [ ]:
# Column names per CMAPSS documentation
COLUMNS = (
    ['engine_id', 'cycle', 'op_set_1', 'op_set_2', 'op_set_3'] +
    [f'sensor_{i}' for i in range(1, 22)]
)

def load_cmapss(subset='FD001'):
    train = pd.read_csv(
        DATA_DIR / f'train_{subset}.txt',
        sep=r'\s+', header=None, names=COLUMNS
    )
    test = pd.read_csv(
        DATA_DIR / f'test_{subset}.txt',
        sep=r'\s+', header=None, names=COLUMNS
    )
    rul_test = pd.read_csv(
        DATA_DIR / f'RUL_{subset}.txt',
        sep=r'\s+', header=None, names=['rul']
    )
    return train, test, rul_test

train, test, rul_test = load_cmapss('FD001')
print(f'Train shape: {train.shape}')
print(f'Test  shape: {test.shape}')
print(f'RUL   shape: {rul_test.shape}')
train.head()

In [ ]:
# ---- Compute RUL for training set ----
max_cycle = train.groupby('engine_id')['cycle'].max().rename('max_cycle')
train = train.join(max_cycle, on='engine_id')
train['rul'] = train['max_cycle'] - train['cycle']
train.drop(columns='max_cycle', inplace=True)

print('RUL stats (uncapped):')
print(train['rul'].describe())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train.groupby('engine_id')['rul'].max(), bins=30, color='#00d4ff', edgecolor='white', alpha=0.8)
ax.set_title('Distribution of Max RUL per Engine (Training Set)', fontsize=14)
ax.set_xlabel('Max RUL (cycles)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../data/processed/rul_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Sensor variance analysis — identify sensors to DROP ----
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
variances = train[sensor_cols].var().sort_values()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#f59e0b' if v < 0.01 else '#00d4ff' for v in variances]
variances.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0.01, color='white', linestyle='--', linewidth=1, label='Drop threshold (var < 0.01)')
ax.set_title('Sensor Variance — Low Variance = Constant = Drop', fontsize=14)
ax.set_xlabel('Variance')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/sensor_variance.png', dpi=150, bbox_inches='tight')
plt.show()

DROP_SENSORS = list(variances[variances < 0.01].index)
print(f'Sensors to DROP (near-zero variance): {DROP_SENSORS}')
# Expected for FD001: sensor_1, sensor_5, sensor_6, sensor_10, sensor_16, sensor_18, sensor_19

In [ ]:
# ---- Correlation heatmap of remaining sensors ----
keep_sensors = [s for s in sensor_cols if s not in DROP_SENSORS]
corr = train[keep_sensors + ['rul']].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Sensor Correlation Matrix (kept sensors + RUL)', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Plot sensor degradation for 5 sample engines ----
sample_engines = train['engine_id'].unique()[:5]
high_corr_sensors = corr['rul'].drop('rul').abs().nlargest(6).index.tolist()

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, sensor in zip(axes, high_corr_sensors):
    for eng in sample_engines:
        subset = train[train['engine_id'] == eng]
        ax.plot(subset['rul'].values[::-1], subset[sensor].values,
                alpha=0.7, linewidth=1)
    ax.set_title(sensor.replace('_', ' ').title())
    ax.set_xlabel('RUL (cycles)')
    ax.invert_xaxis()

fig.suptitle('Top 6 Sensors by RUL Correlation — Degradation Trends', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/sensor_degradation_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Summary printout ----
print('=== EDA Summary ===')
print(f'Training engines: {train["engine_id"].nunique()}')
print(f'Test engines:     {test["engine_id"].nunique()}')
print(f'Total sensor cols: {len(sensor_cols)}')
print(f'Sensors to drop:   {len(DROP_SENSORS)} → {DROP_SENSORS}')
print(f'Sensors to keep:   {len(keep_sensors)} → {keep_sensors}')
print(f'Max RUL:  {train["rul"].max()}')
print(f'Mean RUL: {train["rul"].mean():.1f}')
print(f'High-correlation sensors with RUL:\n{corr["rul"].drop("rul").abs().nlargest(10)}')